# Generar embeddings por palabra (en Colab)

Genera `embeddings_palabras.npy` y `etiquetas_palabras.csv` a partir de **`instancias_final.csv`** (Paenza + conferencias + subtítulos + dataset sintético + una selección de Wikinoticias; lo arma `src/armar_instancias_final.py`, ver `datasets/instancias_final/README.md`).

El embedding de cada palabra es el promedio de los vectores de sus tokens en la capa de entrada de BERT (no se ejecuta la red), así que **no hace falta GPU**: alcanza con un runtime de CPU.

**Salida esperada:** unas ~624.000 palabras × 768 dimensiones (~1,9 GB en float32), de ~46.000 instancias.

**Pasos:** 1) montar Drive y entrar a la carpeta con los archivos, 2) generar, 3) guardar el resultado en esa misma carpeta (junto con `embeddings_meta.json`, que anota con qué versión de `instancias_final.csv` se generó).

## 1. Montar Drive y entrar a la carpeta con los archivos

Cambiá `CARPETA` si no es `MyDrive/TP_AA`. Tiene que contener `instancias_final.csv` (suelto, no en subcarpeta), `armar_instancias.py`, `embeddings_palabras.py` y `rutas.py` (el de Colab, ver `colab/rutas.py`). Los resultados se guardan ahí mismo.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CARPETA = "/content/drive/MyDrive/TP_AA"
%cd {CARPETA}

import os, sys
sys.path.insert(0, CARPETA)   # para poder importar los .py de la carpeta
print(os.listdir("."))

faltan = {"instancias_final.csv", "armar_instancias.py", "embeddings_palabras.py", "rutas.py"} - set(os.listdir("."))
assert not faltan, f"Faltan archivos en {CARPETA}: {faltan}"

## 2. Generar los embeddings

In [ ]:
import time
import numpy as np
from embeddings_palabras import (cargar_instancias, ruta_instancias_final, guardar_textos,
                                 palabras_y_etiquetas, cargar_bert, armar_X_y, agregar_metadatos)

t0 = time.time()
RUTA = ruta_instancias_final()              # instancias_final.csv (ya viene filtrado: no se vuelve a filtrar)
df = cargar_instancias(RUTA)
guardar_textos(df)
print("instancias:", len(df), "\n", df["fuente"].value_counts().to_string())

instancias = {int(i): palabras_y_etiquetas(t) for i, t in zip(df["Indice"], df["Fragmento"])}
print("palabras en total:", sum(map(len, instancias.values())))

tokenizer, matriz = cargar_bert()
X, etiquetas = armar_X_y(instancias, tokenizer, matriz)
etiquetas = agregar_metadatos(etiquetas, df)      # suma fuente, clase_instancia y bloque (al final)
print("X:", X.shape, X.dtype, f"({X.nbytes / 1e9:.2f} GB)", "| etiquetas:", etiquetas.shape)
print(f"tiempo: {time.time() - t0:.0f} s")

### Verificación

X y las etiquetas tienen que tener la misma cantidad de filas, sin NaN, y el `instancia_id` tiene que ser el `Indice` original. Además, todas las instancias del CSV tienen que tener al menos una palabra.

In [ ]:
from embeddings_palabras import verificar_run

verificar_run(X, etiquetas, df)
print("instancias distintas:", etiquetas["instancia_id"].nunique(), "de", len(df))
print(etiquetas.head())

## 3. Guardar

Como el directorio de trabajo ya es la carpeta de Drive, los archivos quedan guardados ahí y no se pierden al reiniciar el runtime.

In [ ]:
from embeddings_palabras import guardar_run

guardar_run(X, etiquetas, RUTA, carpeta=".")      # .npy + .csv + embeddings_meta.json
print(os.listdir("."))

### Para usarlos después en `lstm_puntuacion.ipynb`

Antes de la celda de datos:

```python
from google.colab import drive
drive.mount("/content/drive")
%cd /content/drive/MyDrive/TP_AA
```